In [1]:
# ---- Diagnostic: January wind-infeasibility root cause ----
# Sweeps C_AW across the ITTC-consistent calibration band (target ratios
# 0.15 / 0.20 / 0.25 at Hs=3m/Tp=9s/head/service-speed) and checks whether
# the wind-inclusive DP is feasible under DEADLINE_H, for both Jan and Jul.
# All C_AW variants below are LOCAL to this cell -- src/ is untouched.

import pandas as pd
from scipy.optimize import brentq

from src.models.ship import SHIP
from src.models.engine import Engine, SMCR_KW
from src.models.propeller import T_FRAC
from src.models.waves import added_resistance_waves, C_AW_CALIBRATED, TARGET_RATIO_AT_HS3
from src.models.resistance import R_calm_cond, thrust_from_prop, torque_from_prop, R_wind
from src.optimizer.dp import run_dp_optimizer, compute_deadline_h
from src.optimizer.leg_cost import RPM_CANDIDATES, BALLAST_CONDITION, leg_weather, leg_wind
from src.data.era5 import load_era5, extract_route_weather
from src.database.connection import get_connection
from src.database.queries import get_route_legs

_ENGINE = Engine('ISO')
ETA_SHAFT = 0.98


# --- local, C_AW-explicit versions (diagnostic only, not src/) ---

def _solve_equilibrium_speed_cond_diag(rpm, Hs, Tp, wave_dir, heading, ballast, C_AW,
                                        u10=0.0, v10=0.0, t=T_FRAC, V_range=(0.5, 25.0)):
    def imbalance(V_ms):
        V_kn = V_ms / 0.514444
        R_total = (R_calm_cond(V_ms, ballast)
                   + added_resistance_waves(Hs, Tp, V_kn, wave_dir, heading, C_AW=C_AW)
                   + R_wind(u10, v10, V_kn, heading, ballast))
        return thrust_from_prop(V_ms, rpm) * (1 - t) - R_total
    lo, hi = V_range
    if imbalance(lo) * imbalance(hi) > 0:
        return None
    return brentq(imbalance, lo, hi, xtol=1e-4)


def _fuel_rate_diag(rpm, Hs, Tp, wave_dir, heading, ballast, C_AW, u10=0.0, v10=0.0):
    V_eq = _solve_equilibrium_speed_cond_diag(rpm, Hs, Tp, wave_dir, heading, ballast, C_AW, u10, v10)
    if V_eq is None:
        return None, None
    Q = torque_from_prop(V_eq, rpm)
    P_brake_kw = Q * (rpm * 2 * 3.141592653589793 / 60.0) / ETA_SHAFT / 1000.0
    load_pct = 100 * P_brake_kw / SMCR_KW
    load_clamped = max(_ENGINE.load_pct.min(), min(_ENGINE.load_pct.max(), load_pct))
    sfoc = float(_ENGINE.sfoc_of_load(load_clamped))
    return V_eq / 0.514444, sfoc * P_brake_kw / 1e6 * 1000.0  # kg/h


def _build_cost_table_diag(legs, summary, month_period, ballast, C_AW, include_wind):
    rows = []
    for leg_id in legs['leg_id']:
        dist_nm = legs.loc[leg_id, 'dist_nm']
        heading = legs.loc[leg_id, 'course_deg']
        Hs, Tp, wave_dir = leg_weather(legs, summary, leg_id, month_period)
        if include_wind:
            wind_speed, wind_dir_from = leg_wind(legs, summary, leg_id, month_period)
            dir_to_rad = ((wind_dir_from + 180.0) % 360.0) * 3.141592653589793 / 180.0
            u10, v10 = wind_speed * __import__('numpy').sin(dir_to_rad), wind_speed * __import__('numpy').cos(dir_to_rad)
        else:
            u10, v10 = 0.0, 0.0
        for rpm in RPM_CANDIDATES:
            V_kn, fuel_kgh = _fuel_rate_diag(rpm, Hs, Tp, wave_dir, heading, ballast, C_AW, u10, v10)
            if V_kn is None or V_kn <= 0.5:
                continue
            time_h = dist_nm / V_kn
            rows.append({'leg_id': leg_id, 'rpm': rpm, 'time_h': time_h,
                         'fuel_t': fuel_kgh * time_h / 1000.0, 'V_kn': V_kn})
    return pd.DataFrame(rows)


def _leg_min_time(cost_table, leg_id):
    """Fastest achievable time_h for this leg at any candidate rpm."""
    grp = cost_table[cost_table['leg_id'] == leg_id]
    return grp['time_h'].min() if not grp.empty else float('inf')


def _wind_variance_flag(summary, legs, leg_id, month_period, use_point='to_pt'):
    pt = legs.loc[leg_id, use_point]
    row = summary[(summary['leg_point'] == pt) & (summary['datetime'] == month_period)]
    return float(row['wind_dir_variance_flag'].iloc[0]) if not row.empty else float('nan')


# --- run the sweep ---

conn = get_connection()
legs = get_route_legs(conn, 'rotterdam_ny')
conn.close()

weather_df = load_era5()
summary = extract_route_weather(weather_df)
deadline_h = compute_deadline_h(legs['dist_nm'].sum(), SHIP['V_service_kn'])

TARGET_RATIOS = [0.15, 0.20, 0.25]
MONTHS = {'2025-01': 'January (harsh)', '2025-07': 'July (calm)'}

print(f"DEADLINE_H = {deadline_h:.2f} h  |  total distance = {legs['dist_nm'].sum():.1f} nm\n")

for period, label in MONTHS.items():
    month_period = pd.Period(period, freq='M')
    print("=" * 78)
    print(f"{label}  ({period})")
    print("=" * 78)

    for ratio in TARGET_RATIOS:
        c_aw = C_AW_CALIBRATED * (ratio / TARGET_RATIO_AT_HS3)
        cost_table = _build_cost_table_diag(legs, summary, month_period, BALLAST_CONDITION,
                                             C_AW=c_aw, include_wind=True)

        policy_df, dp_fuel, dp_time = run_dp_optimizer(cost_table, legs, deadline_h)
        feasible = pd.notna(dp_fuel)

        status = "FEASIBLE" if feasible else "INFEASIBLE"
        print(f"\n  -- ratio={ratio:.2f} (C_AW={c_aw:.3f}) : {status} --")

        if feasible:
            print(f"     total_time={dp_time:.2f}h (budget {deadline_h:.2f}h)  "
                  f"total_fuel={dp_fuel:.1f}t")
            continue

        # Infeasible: show per-leg bottleneck diagnostics
        print(f"     {'leg':>4} {'min_time_h':>11} {'fair_share_h':>13} {'overrun_h':>10} {'wind_var_flag':>14}")
        n_legs = len(legs)
        fair_share = deadline_h / n_legs
        for leg_id in legs['leg_id']:
            min_t = _leg_min_time(cost_table, leg_id)
            overrun = min_t - fair_share
            flag = _wind_variance_flag(summary, legs, leg_id, month_period)
            marker = "  <-- bottleneck" if overrun > 0 else ""
            print(f"     {leg_id:>4} {min_t:>11.3f} {fair_share:>13.3f} {overrun:>+10.3f} "
                  f"{flag:>14.3f}{marker}")

    print()

DEADLINE_H = 202.47 h  |  total distance = 3149.5 nm

January (harsh)  (2025-01)
CANARY: relative_wind called, angle=57.7
CANARY: relative_wind called, angle=12.3
CANARY: relative_wind called, angle=57.7
CANARY: relative_wind called, angle=12.3
CANARY: relative_wind called, angle=48.2
CANARY: relative_wind called, angle=31.9
CANARY: relative_wind called, angle=38.3
CANARY: relative_wind called, angle=37.2
CANARY: relative_wind called, angle=37.0
CANARY: relative_wind called, angle=37.0
CANARY: relative_wind called, angle=37.0
CANARY: relative_wind called, angle=57.7
CANARY: relative_wind called, angle=12.3
CANARY: relative_wind called, angle=57.7
CANARY: relative_wind called, angle=12.3
CANARY: relative_wind called, angle=47.5
CANARY: relative_wind called, angle=31.3
CANARY: relative_wind called, angle=37.7
CANARY: relative_wind called, angle=36.5
CANARY: relative_wind called, angle=36.4
CANARY: relative_wind called, angle=36.4
CANARY: relative_wind called, angle=36.4
CANARY: relative_

In [2]:
from src.optimizer.leg_cost import build_cost_table, leg_cost
from src.optimizer.dp import run_dp_optimizer
from src.models.propulsion import fuel_rate_from_equilibrium_cond

In [ ]:
# ---- Diagnostic: wind direction sign-convention check ----
# Cell 77 in the original notebook started this check but was truncated
# before a pass/fail conclusion (per the dependency map). This reconstructs
# it: for controlled scenarios with a known physically-correct answer,
# verify that wind_dir_from -> (u10,v10) -> relative_wind() produces the
# expected relative angle and R_wind ordering.
#
# NOTE: the original version of this check hardcoded "crosswind should be
# near 90 deg apparent" -- that ignores the ship's own forward velocity,
# which always drags the apparent wind angle forward of the true beam for
# a moving vessel. Fixed by computing the expected angle independently
# (expected_apparent_angle) instead of assuming a fixed numbegr.

import numpy as np
from src.models.resistance import relative_wind, R_wind, A_T, C_AA

SHIP_SPEED_KN = 16.8   # arbitrary steady speed for the test
WIND_SPEED_MS = 15.0   # arbitrary fixed wind magnitude
HEADING_DEG = 0.0      # ship steaming due north
BALLAST = 'laden'

def dir_from_to_uv(wind_dir_from_deg, wind_speed_ms):
    """Same conversion used in leg_cost.py: met convention 'direction wind
    is coming FROM' -> (u10, v10) vector wind is blowing TOWARD."""
    dir_to_rad = np.radians((wind_dir_from_deg + 180.0) % 360.0)
    u10 = wind_speed_ms * np.sin(dir_to_rad)
    v10 = wind_speed_ms * np.cos(dir_to_rad)
    return u10, v10

def expected_apparent_angle(wind_dir_from_deg, wind_speed_ms, ship_speed_kn, heading_deg):
    """Reference calculation, independent of relative_wind(), to sanity-check
    the apparent wind angle including the ship's own forward-motion shift."""
    heading_rad = np.radians(heading_deg)
    ship_speed_ms = ship_speed_kn * 0.514444
    ship_vx = ship_speed_ms * np.sin(heading_rad)
    ship_vy = ship_speed_ms * np.cos(heading_rad)

    u10, v10 = dir_from_to_uv(wind_dir_from_deg, wind_speed_ms)

    rel_u = u10 - ship_vx
    rel_v = v10 - ship_vy
    bearing = np.degrees(np.arctan2(-rel_u, -rel_v)) % 360
    return np.abs(((bearing - heading_deg + 180) % 360) - 180)

# Scenarios: ship heading due north (0 deg).
# - "headwind": wind FROM the north (dir_from=0) -> blows straight at the bow
# - "tailwind": wind FROM the south (dir_from=180) -> blows straight at the stern
# - "crosswind": wind FROM the east (dir_from=90) -> hits the beam (true beam;
#   apparent angle will read forward of 90 due to ship's own speed -- expected)
scenarios = [
    ("Headwind (wind FROM north, bow-on)",   0.0,   "should be near 0 deg (bow) -- MAX windage resistance expected"),
    ("Crosswind (wind FROM east, beam-on)",  90.0,  "apparent angle shifts forward of 90 deg due to ship's own speed"),
    ("Tailwind (wind FROM south, stern-on)", 180.0, "should be near 180 deg (stern) -- MIN/thrust-assisting expected"),
]

print(f"{'Scenario':<42} {'angle_from_bow':>15} {'C_AA':>7} {'A_T(m2)':>9} {'R_wind(N)':>11}  expectation")
print("-" * 115)

results = {}
for label, wind_dir_from, note in scenarios:
    u10, v10 = dir_from_to_uv(wind_dir_from, WIND_SPEED_MS)
    V_rel, angle = relative_wind(u10, v10, SHIP_SPEED_KN, HEADING_DEG)
    Cd = C_AA(angle)
    area = A_T(angle, BALLAST)
    r_wind = R_wind(u10, v10, SHIP_SPEED_KN, HEADING_DEG, BALLAST)
    results[label] = (angle, r_wind, wind_dir_from)
    print(f"{label:<42} {angle:>15.1f} {Cd:>7.3f} {area:>9.1f} {r_wind:>11.1f}  {note}")

print("-" * 115)

# ---- Verdict ----
head_angle, head_R, head_dir_from = results["Headwind (wind FROM north, bow-on)"]
cross_angle, cross_R, cross_dir_from = results["Crosswind (wind FROM east, beam-on)"]
tail_angle, tail_R, tail_dir_from = results["Tailwind (wind FROM south, stern-on)"]

expected_head = expected_apparent_angle(head_dir_from, WIND_SPEED_MS, SHIP_SPEED_KN, HEADING_DEG)
expected_cross = expected_apparent_angle(cross_dir_from, WIND_SPEED_MS, SHIP_SPEED_KN, HEADING_DEG)
expected_tail = expected_apparent_angle(tail_dir_from, WIND_SPEED_MS, SHIP_SPEED_KN, HEADING_DEG)

TOL_DEG = 0.5
angle_ok = (abs(head_angle - expected_head) < TOL_DEG
            and abs(cross_angle - expected_cross) < TOL_DEG
            and abs(tail_angle - expected_tail) < TOL_DEG)
resistance_ok = head_R > tail_R  # head-on windage should exceed following-wind windage

print(f"\nAngle convention check: headwind->{head_angle:.1f} deg (expected {expected_head:.1f}), "
      f"crosswind->{cross_angle:.1f} deg (expected {expected_cross:.1f}), "
      f"tailwind->{tail_angle:.1f} deg (expected {expected_tail:.1f})")
print("  ", "PASS -- matches physically-correct apparent-wind geometry" if angle_ok
      else "FAIL -- angle convention looks INVERTED or scrambled")

print(f"\nResistance ordering check: R_wind(head)={head_R:.1f} N vs R_wind(tail)={tail_R:.1f} N")
print("  ", "PASS -- headwind resistance > tailwind resistance, as physically expected" if resistance_ok
      else "FAIL -- headwind resistance is NOT greater than tailwind -- C_AA/A_T logic likely inverted")

print("\nOVERALL:", "PASS" if (angle_ok and resistance_ok) else "FAIL -- sign/angle convention bug confirmed")

Scenario                                    angle_from_bow    C_AA   A_T(m2)   R_wind(N)  expectation
-------------------------------------------------------------------------------------------------------------------
CANARY: relative_wind called, angle=0.0
CANARY: relative_wind called, angle=0.0
Headwind (wind FROM north, bow-on)                     0.0   0.850     574.6    167223.9  should be near 0 deg (bow) -- MAX windage resistance expected
CANARY: relative_wind called, angle=60.1
CANARY: relative_wind called, angle=60.1
Crosswind (wind FROM east, beam-on)                   60.1   0.658    2249.8    271934.1  apparent angle shifts forward of 90 deg due to ship's own speed
CANARY: relative_wind called, angle=180.0
CANARY: relative_wind called, angle=180.0
Tailwind (wind FROM south, stern-on)                 180.0   0.085     574.6      1209.1  should be near 180 deg (stern) -- MIN/thrust-assisting expected
----------------------------------------------------------------------------